# Stage 7 · Vehicle pipeline review

Camera or video → YOLO26s → temporal association → duplicate suppression → evidence frame → location and severity → JSONL and CSV.

This notebook reads one archived run. The footage and predictor type are recorded with each measurement.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from roadguard.utils.paths import project_root

root = project_root()
selected = os.environ.get("ROADGUARD_STAGE7_RUN")
if selected:
    run = Path(selected)
    result_path = run / "stage7_result.json"
else:
    latest = root / "artifacts/experiments/stage7_user_video_result.json"
    result_path = latest if latest.is_file() else latest.with_name("stage7_result.json")
    run = root / json.loads(result_path.read_text(encoding="utf-8"))["run"]
result = json.loads(result_path.read_text(encoding="utf-8"))
metadata = json.loads((run / "metadata.json").read_text(encoding="utf-8"))
print(result["source_provenance"])
print("Measured device:", metadata["identity"]["device"])

## Sequential performance

Each measured interval includes acquisition, preprocessing, inference, association, writing and, where applicable, playback pacing. The mean processing time excludes intentional waiting. Synthetic benchmark warm-up is recorded separately. The user-video run includes the first predictor call.

In [ ]:
runtime = pd.DataFrame(result["runtime"]).T
columns = [
    "predictor",
    "source_fps",
    "processed_frames",
    "software_skipped_frames",
    "mean_detector_ms",
    "mean_total_frame_ms_excluding_intentional_wait",
    "effective_fps",
    "realtime_late_frames",
    "warmup_ms_excluded",
]
display(runtime[columns].round(3))
print("Precision:", metadata["predict_precision"])
print("Source kind:", result["source_kind"])

## Event reduction and saved evidence

The scripted fixture uses known observations to test event merging after a short gap, separation of simultaneous defects, and suppression of one short-lived observation. Its throughput measures the fixture only. The annotated video uses temporary track IDs, and JSONL/CSV use final event IDs.

In [ ]:
columns = [
    "raw_detections",
    "final_events",
    "merged_detections",
    "merged_track_fragments",
    "discarded_short_track_detections",
    "duplicate_reduction_ratio",
    "location_mode",
    "severity",
]
display(runtime[columns].round(4))
for mode in runtime.index:
    path = run / mode / "events.jsonl"
    events = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    for event in events[:2]:
        print(mode, event["class_id"], event["detection_count"], "observations")
        print(event["event_id"], event["location"]["source"], event["severity"])
        display(Image(filename=str(run / mode / event["evidence_frame_path"]), width=620))

## Field-validation status

The synthetic and scripted outputs are archived independently. The supplied road clip exercises full-resolution real-video processing. Vehicle hardware, synchronized GPS and sustained camera throughput still require validation.

In [ ]:
print("Field validation:", result["field_validation_status"])
print("Vehicle hardware validation:", result.get("vehicle_hardware_validation_status", "pending"))
print("Output formats and video verified:", result["output_verified"])
print("Training occurred:", result["integrity"]["training_occurred"])
print("Stage 6 images reused:", result["integrity"]["stage6_test_reused_for_field_validation"])
print("Checkpoint SHA-256:", result["integrity"]["checkpoint_sha256"])

## Independent road-video observations

Sampled frames and saved predictions illustrate continuity and gaps. The clip has no reference annotations.

In [ ]:
if "video_source" in result:
    media = result["video_source"]
    display(
        pd.Series(
            {
                key: media[key]
                for key in (
                    "original_filename",
                    "codec_fourcc",
                    "fps",
                    "total_frames",
                    "duration_seconds",
                    "sha256",
                )
            },
            name="Verified attachment",
        ).to_frame()
    )
    stats = result["runtime"]["throughput"]
    display(
        pd.DataFrame(
            {
                "Detections": stats["detections_by_class"],
                "Final events": stats["events_by_class"],
            }
        )
    )
    print("Temporal groups:", stats["temporal_groups"])
    print("Linked observations:", stats["linked_observations"])
    print("Adjacent-frame links:", stats["adjacent_frame_links"])
    for note in result["visual_observations"]["observations"]:
        print(note["topic"], note["finding"])
    for name in result["visual_observations"]["review_images"]:
        display(Image(filename=str(run / name), width=1000))

## Stage 7.1 · Event identity and SQLite

Replay uses frozen frame predictions. The three-second cap is a conservative association boundary. Physical defect identity still requires reference annotations.

In [ ]:
identity_run = os.environ.get("ROADGUARD_STAGE71_RUN")
if identity_run:
    identity_archive = Path(identity_run)
else:
    identity_pointer = root / "artifacts/experiments/stage7_1_result.json"
    identity_archive = (
        root / json.loads(identity_pointer.read_text(encoding="utf-8"))["run"]
        if identity_pointer.is_file()
        else None
    )

if identity_archive is not None:
    identity = json.loads(
        (identity_archive / "event_identity_analysis.json").read_text(encoding="utf-8")
    )
    database = json.loads(
        (identity_archive / "database_consistency.json").read_text(encoding="utf-8")
    )
    columns = ["filename", "raw_detections", "baseline_events", "replay_events", "boundary_count"]
    display(pd.DataFrame(identity["inputs"])[columns])
    display(pd.DataFrame(database["sessions"])[["video", "jsonl_rows", "csv_rows", "sqlite_rows"]])
    print("SQLite integrity:", database["integrity_check"])
    print("Duplicate inserts blocked:", database["duplicate_attempts_blocked"])
    print("Physical over-merges confirmed:", identity["physical_over_merges_confirmed"])
    long_tracks = [
        row
        for row in identity["tracks_and_events"]
        if row["variant"] == "baseline" and row["kind"] == "track" and "long_span" in row["flags"]
    ]
    for row in long_tracks:
        print(row["event_id"], row["duration_frames"], "frames")
        display(Image(filename=row["contact_sheet"], width=1000))

## Stage 7.2 · Physical defect identity

The review prioritizes flagged, adjacent and duration-limited events from two road clips.
The metrics describe manually adjudicated correspondences in this sample. Ambiguous cases
are shown separately and excluded from the rate denominators.

GT IDs refer to visually identifiable physical defects. Event UUIDs link each judgment
to the stored event. The images show the same road fork assigned to two events and the
archived frames around both T48 split boundaries.

In [ ]:
review_run = os.environ.get("ROADGUARD_STAGE72_RUN")
review_pointer = root / "artifacts/experiments/stage7_2_result.json"
if review_run:
    review_archive = Path(review_run)
elif review_pointer.is_file():
    review_archive = root / json.loads(review_pointer.read_text(encoding="utf-8"))["run"]
else:
    review_archive = None

if review_archive is not None:
    review_metrics = pd.read_csv(
        review_archive / "event_identity_metrics.csv", dtype=str, keep_default_na=False
    )
    display(review_metrics[review_metrics["metric"] != "scope"])
    review_events = pd.read_csv(review_archive / "event_identity_ground_truth.csv")
    display(
        review_events[
            ["case_id", "start_frame", "end_frame", "ground_truth_defect_ids", "error_type"]
        ]
    )
    t48 = json.loads((review_archive / "t48_review.json").read_text(encoding="utf-8"))
    print("Historical T48:", t48["baseline_classification"])
    print("Physical identity effect of the duration cap:", t48["safeguard_effect"])
    for observation in t48["visual_evidence"]:
        print(observation)
    display(Image(filename=str(review_archive / t48["boundary_sheet"]), width=1000))
    fork = review_archive / "figures/contact_sheets/details/shared_fork_v1_t69_t70.jpg"
    if fork.is_file():
        display(Image(filename=str(fork), width=1000))